# 12-02 Agent 设计面试题

**高频考点**: Agent 架构、ReAct、记忆系统、多 Agent、安全、工具选择

---

In [ ]:
# Q1: Agent 核心架构
print("""
Q: 一个完整的 Agent 系统包含哪些组件？

A: Agent = LLM (大脑) + Memory (记忆) + Tools (工具) + Planning (规划)

  ┌─────────────────────────────────────┐
  │              Agent                   │
  │  ┌──────────┐  ┌──────────────┐     │
  │  │  LLM     │  │   Memory     │     │
  │  │ (推理核心)│  │ 短期/长期/向量│     │
  │  └────┬─────┘  └──────────────┘     │
  │       │                              │
  │  ┌────┴─────┐  ┌──────────────┐     │
  │  │ Planning │  │   Tools      │     │
  │  │ ReAct/P&E│  │ API/DB/搜索  │     │
  │  └──────────┘  └──────────────┘     │
  └─────────────────────────────────────┘
  
  关键设计决策:
  1. LLM 选择: GPT-4(强推理) vs Qwen-7B(低成本) vs 混合(路由)
  2. 记忆策略: 对话历史(短期) + 用户画像(长期) + 知识库(向量)
  3. 工具设计: 工具描述要清晰、参数要有类型约束
  4. 规划策略: 简单任务→ReAct, 复杂任务→Plan-and-Execute
""")

In [ ]:
# Q2: ReAct vs Function Calling
print("""
Q: ReAct 和 Function Calling 的区别？

A:
  ReAct (2022, Yao et al.):
    - Prompt-based: 在 prompt 中要求 LLM 输出 Thought/Action/Observation
    - 格式: 自由文本，需要解析
    - 兼容性: 任何 LLM 都能用
    - 缺点: 格式不稳定，需要 few-shot 示例
  
  Function Calling (OpenAI, 2023):
    - API-native: 模型经过训练，原生支持工具调用
    - 格式: 结构化 JSON，{"name": "tool", "arguments": {...}}
    - 兼容性: 需要模型原生支持 (GPT-4/Claude/Qwen)
    - 优点: 格式稳定、支持并行调用、参数类型校验
  
  实际选择:
    - 优先 Function Calling (格式稳定)
    - 老模型/开源模型回退到 ReAct
    - LangChain 的 tool_calling agent 自动适配

Q: Agent 的安全风险有哪些？

A:
  1. Prompt Injection: 用户输入中注入恶意指令
     → 防御: 输入过滤、角色隔离、LLM Guard
  2. Tool Abuse: Agent 被诱导执行危险操作
     → 防御: 工具权限最小化、操作确认、沙箱
  3. 信息泄露: Agent 泄露 System Prompt 或内部数据
     → 防御: Prompt 保护、输出过滤
  4. 死循环/资源耗尽: Agent 无限循环调用工具
     → 防御: 最大步数限制、超时机制、成本上限
""")

In [ ]:
# Q3: 记忆系统设计
print("""
Q: 如何为 Agent 设计记忆系统？

A: 三层记忆架构

  1. 短期记忆 (Working Memory)
     - 当前对话历史 (最近 N 轮)
     - 实现: deque(maxlen=10) 或 token 截断
     - 挑战: context window 有限
     
  2. 长期记忆 (Long-term Memory)
     - 用户画像、偏好、历史总结
     - 实现: 数据库存储 (Redis/MySQL)
     - 写入: 对话结束时提取关键信息
     - 读取: 对话开始时加载到 System Prompt
     
  3. 向量记忆 (Semantic Memory)
     - 历史对话/文档的语义索引
     - 实现: 向量数据库 (FAISS/Chroma/Milvus)
     - 写入: Embedding 后存储
     - 读取: 相似度检索 top-k

  记忆压缩策略:
  - 滑动窗口: 只保留最近 N 轮
  - 摘要压缩: 用 LLM 总结历史对话
  - Token 截断: 超出时从最早的消息开始删
  - 重要性排序: 用 LLM 判断哪些信息重要

  B站广告场景:
  - 短期: 当前咨询的多轮对话
  - 长期: 广告主的行业/预算/历史投放偏好
  - 向量: 广告知识库文档检索
""")

In [ ]:
# Q4: 多 Agent 编排
print("""
Q: 多 Agent 系统有哪些编排模式？

A:
  1. Pipeline (流水线):
     Agent_A → Agent_B → Agent_C
     适合: 固定流程 (数据清洗→分析→报告)
     
  2. Supervisor (主管模式):
     Supervisor 根据任务分配给子 Agent
     适合: 动态任务分配 (客服路由)
     LangGraph 推荐模式
     
  3. Debate (辩论):
     多个 Agent 相互质疑、改进
     适合: 需要批判性思考 (代码审查)
     
  4. Voting (投票):
     多个 Agent 独立生成，投票选最佳
     适合: 高质量要求 (关键决策)
     
  5. Hierarchical (层级):
     Manager → Team Lead → Worker
     适合: 大型复杂任务分解
  
  Agent 间通信:
  - 共享状态 (LangGraph State): 通过 TypedDict 传递
  - 消息传递 (AutoGen): Agent 间直接发消息
  - 黑板模式: 公共数据区域，Agent 自由读写

Q: 如何避免多 Agent 系统的常见问题？

A:
  1. 无限循环: 设 max_iterations + 终止条件
  2. 责任模糊: 每个 Agent 有清晰的 System Prompt
  3. 信息丢失: 用结构化 State 而非自由文本
  4. 成本爆炸: 监控 token 用量，设预算上限
""")

In [ ]:
# Q5: Agent 工具选择策略
print("""
Q: Agent 如何选择合适的工具？

A: 四种常见策略

  1. 关键词匹配 (Keyword Matching):
     - 用户输入中匹配工具描述关键词
     - 优点: 快速、零成本
     - 缺点: 语义理解差，同义词处理弱
     
  2. Embedding 相似度:
     - 将用户输入和工具描述都 embed，计算余弦相似度
     - 优点: 语义理解强
     - 缺点: 需要 Embedding 模型，有延迟
     
  3. LLM 路由:
     - 让 LLM 根据用户意图选择工具 (Function Calling)
     - 优点: 最灵活、理解力最强
     - 缺点: 成本高、有延迟
     
  4. 混合策略 (推荐):
     - 第一步: 关键词/Embedding 粗筛 (从100个工具→5个)
     - 第二步: LLM 精选 (从5个→1-2个)
     - 优点: 兼顾成本和效果

  B站广告场景工具选择:
  - "查看投放数据" → 数据查询 Tool
  - "创建广告计划" → 投放管理 Tool  
  - "审核广告素材" → 内容审核 Tool
  - 模糊意图 → LLM 路由判断
""")

# Q6: Agent 实际落地踩坑经验
print("""
Q: Agent 落地有哪些常见坑？

A:
  1. LLM 输出不稳定:
     → 用 Structured Output / JSON Mode 约束
     → 加 retry + fallback 机制
     
  2. 长对话上下文丢失:
     → 摘要压缩 + 关键信息提取
     → 使用 RAG 代替超长 context
     
  3. 工具调用参数错误:
     → 工具描述要详细、参数加类型和示例
     → 加参数校验层
     
  4. Agent 幻觉严重:
     → 强制引用来源 (RAG 带出处)
     → 加 fact-checking Agent
     
  5. 多 Agent 协作混乱:
     → 每个 Agent 职责单一、边界清晰
     → 用结构化 State 而非自由文本传递信息
     → 设置 max_iterations 防止死循环
""")

## 面试速查卡片

| 题目 | 一句话回答 |
|------|-----------|
| Agent 核心组件 | LLM + Memory + Tools + Planning |
| ReAct vs Function Calling | ReAct 是 prompt-based 通用方案; FC 是 API-native 结构化方案，优先选 FC |
| 三层记忆架构 | 短期(对话历史) + 长期(用户画像/DB) + 向量(语义检索) |
| 多 Agent 编排模式 | Pipeline / Supervisor / Debate / Voting / Hierarchical |
| Agent 安全风险 | Prompt Injection / Tool Abuse / 信息泄露 / 死循环 |
| 工具选择策略 | 关键词粗筛 → Embedding 召回 → LLM 精选 (混合策略) |
| Agent 落地踩坑 | 输出不稳定→结构化约束; 幻觉→RAG+引用; 死循环→max_iterations |
| B站场景 Agent 设计 | 客服(RAG+记忆) / 素材生成(LangGraph+审核) / 投放优化(多Agent+监控) |